In [ ]:
# Celda 1: subir el CSV del censo INEGI
from google.colab import files
subido = files.upload()

Saving Copia de Personas_vivienda2010_2015_2020_20VAR_DES.csv to Copia de Personas_vivienda2010_2015_2020_20VAR_DES (2).csv


MODELO A

In [ ]:
import os, time
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix

EDAD_MINIMA = 15
NIVACAD_MIN = 4
NIVACAD_MAX = 11
TEST_SIZE = 0.20
RANDOM_STATE = 42
N_SEMILLAS = 8

VARIABLES_ENTRADA = [
    "ENT", "MUN", "SEXO", "EDAD", "SERSALUD", "PERTE_INDIGENA", "NACIONALIDAD",
    "ASISTEN", "ESCOLARI", "NIVACAD", "ESCOACUM", "SITUA_CONYUGAL",
    "SITUACION_TRAB", "COMPUTADORA", "CELULAR", "INTERNET", "INGTRHOG",
    "TAMLOC", "CPV",
]
VARIABLE_SALIDA = "DES"

RUTA_MODELO_A = "modelo_desercion_mlp.joblib"
RUTA_SCALER_A = "scaler_desercion.joblib"


def cargar_y_limpiar_datos_A(ruta_csv: str) -> pd.DataFrame:
    print(f"Cargando {ruta_csv} ...")
    columnas = VARIABLES_ENTRADA + [VARIABLE_SALIDA]
    df = pd.read_csv(ruta_csv, usecols=columnas, encoding="utf-8-sig")
    print(f"  Registros totales leidos: {len(df):,}")
    df = df[columnas]

    antes = len(df); df = df.dropna()
    print(f"  Eliminados por valores vacios: {antes - len(df):,}")

    antes = len(df); df = df.drop_duplicates()
    print(f"  Eliminados por duplicados: {antes - len(df):,}")

    antes = len(df); df = df[df["EDAD"] >= EDAD_MINIMA]
    print(f"  Eliminados por EDAD >= {EDAD_MINIMA}: {antes - len(df):,}")

    antes = len(df); df = df[(df["NIVACAD"] >= NIVACAD_MIN) & (df["NIVACAD"] <= NIVACAD_MAX)]
    print(f"  Eliminados por NIVACAD [{NIVACAD_MIN}-{NIVACAD_MAX}]: {antes - len(df):,}")

    antes = len(df); df = df[df["ESCOACUM"] != 99]
    print(f"  Eliminados por ESCOACUM == 99: {antes - len(df):,}")

    antes = len(df); df = df[df["INGTRHOG"] < 999998]
    print(f"  Eliminados por INGTRHOG >= 999998: {antes - len(df):,}")

    print(f"  Registros finales: {len(df):,}")
    valores_des = sorted(df[VARIABLE_SALIDA].unique().tolist())
    print(f"  Valores unicos en DES: {valores_des}")
    if set(valores_des) != {0, 5, 9} and set(valores_des) != {1, 3, 9}:
        print("  ADVERTENCIA: los valores de DES no coinciden con lo esperado.")
    return df


df_A = cargar_y_limpiar_datos_A(RUTA_CSV)

Cargando Copia de Personas_vivienda2010_2015_2020_20VAR_DES (1).csv ...
  Registros totales leidos: 3,195,460
  Eliminados por valores vacios: 0
  Eliminados por duplicados: 6,714
  Eliminados por EDAD >= 15: 0
  Eliminados por NIVACAD [4-11]: 2,109,152
  Eliminados por ESCOACUM == 99: 0
  Eliminados por INGTRHOG >= 999998: 194
  Registros finales: 1,079,400
  Valores unicos en DES: [0, 5, 9]


In [ ]:
X = df_A[VARIABLES_ENTRADA].values
y = df_A[VARIABLE_SALIDA].values.astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"\nEntrenamiento: {len(X_train):,} | Prueba: {len(X_test):,}")

scaler = MinMaxScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc = scaler.transform(X_test)

mejor_modelo, mejor_r2 = None, -np.inf
print(f"\nProbando {N_SEMILLAS} inicializaciones aleatorias...")
for i in range(N_SEMILLAS):
    semilla = RANDOM_STATE + i
    modelo = MLPRegressor(
        hidden_layer_sizes=(2, 2, 2, 2), activation="relu", solver="adam",
        alpha=0.001, max_iter=5000, random_state=semilla,
    )
    inicio = time.time()
    modelo.fit(X_train_esc, y_train)
    duracion = time.time() - inicio
    r2 = r2_score(y_test, modelo.predict(X_test_esc))
    print(f"  Semilla {semilla}: {modelo.n_iter_} iteraciones, {duracion:.1f}s, R^2 = {r2:.6f}")
    if r2 > mejor_r2:
        mejor_r2, mejor_modelo = r2, modelo

print(f"\nMejor semilla: R^2 = {mejor_r2:.6f}")


Entrenamiento: 863,520 | Prueba: 215,880

Probando 8 inicializaciones aleatorias...
  Semilla 42: 13 iteraciones, 40.1s, R^2 = -0.000015
  Semilla 43: 15 iteraciones, 39.5s, R^2 = 0.998579
  Semilla 44: 22 iteraciones, 67.5s, R^2 = 0.925976
  Semilla 45: 13 iteraciones, 40.5s, R^2 = -0.000001
  Semilla 46: 15 iteraciones, 39.8s, R^2 = 0.925696
  Semilla 47: 13 iteraciones, 41.9s, R^2 = -0.000000
  Semilla 48: 13 iteraciones, 40.8s, R^2 = -0.000022
  Semilla 49: 25 iteraciones, 66.0s, R^2 = 0.925973

Mejor semilla: R^2 = 0.998579


In [ ]:
valores_validos = np.array(sorted(df_A[VARIABLE_SALIDA].unique().tolist()))
y_pred = mejor_modelo.predict(X_test_esc)
print("\n--- Resultados como regresion ---")
print(f"R^2: {r2_score(y_test, y_pred):.9f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.6f}")

y_pred_clase = valores_validos[np.argmin(np.abs(y_pred[:, None] - valores_validos[None, :]), axis=1)]
print("\n--- Resultados como clasificacion ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_clase):.4%}")
print("Matriz de confusion, orden:", list(valores_validos))
print(confusion_matrix(y_test, y_pred_clase, labels=valores_validos))

joblib.dump(mejor_modelo, RUTA_MODELO_A)
joblib.dump(scaler, RUTA_SCALER_A)
print(f"\nGuardados: {RUTA_MODELO_A}, {RUTA_SCALER_A}")


--- Resultados como regresion ---
R^2: 0.998578748
MSE: 0.009560

--- Resultados como clasificacion ---
Accuracy: 100.0000%
Matriz de confusion, orden: [np.int64(0), np.int64(5), np.int64(9)]
[[178031      0      0]
 [     0  23837      0]
 [     0      0  14012]]

Guardados: modelo_desercion_mlp.joblib, scaler_desercion.joblib


In [ ]:
from google.colab import files
for nombre in ["modelo_desercion_mlp.joblib", "scaler_desercion.joblib", "model_des.pkl", "model_metadata.json"]:
    files.download(nombre)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>